In [9]:
import numpy as np
import pandas as pd
from itertools import combinations
from scipy.optimize import differential_evolution

In [ ]:
theta = {
    "GSL": {"MT": 0.000531, "X": 0.000122, "AUT": 0.000297, "Y": 0.000063},
    "SSL": {"MT": 0.001090, "X": 0.000199, "AUT": 0.000484, "Y": 0.000112}
        }
mu_AUT = 1.2e-8

PAIR_KEYS = [("MT", "Y"), ("MT", "AUT"), ("MT", "X"), ("AUT", "X")]
COMP_ORDER = ["MT", "Y", "X", "AUT"]


# functions

In [11]:
def get_Ne_per_component(dNe):
    if all(k in dNe for k in ["MT", "Y"]):
        NeMT = NeF = dNe["MT"]
        NeY = NeM = dNe["Y"]
        NeX = (9 * NeF * NeM) / (2 * NeF + 4 * NeM)
        NeAUT = (4 * NeF * NeM) / (NeF + NeM)
    elif all(k in dNe for k in ["MT", "AUT"]):
        NeMT = NeF = dNe["MT"]
        NeAUT = dNe["AUT"]
        NeY = NeM = (NeMT * NeAUT) / (4 * NeMT - NeAUT)
        NeX = (9 * NeF * NeM) / (2 * NeF + 4 * NeM)
    elif all(k in dNe for k in ["MT", "X"]):
        NeMT = NeF = dNe["MT"]
        NeX = dNe["X"]
        NeY = NeM = (2 * NeMT * NeX) / (9 * NeMT - 4 * NeX)
        NeAUT = (4 * NeF * NeM) / (NeF + NeM)
    elif all(k in dNe for k in ["AUT", "X"]):
        NeAUT = dNe["AUT"]
        NeX = dNe["X"]
        NeY = NeM = (2 * NeAUT * NeX) / (16 * NeX - 9 * NeAUT)
        NeMT = NeF = (2 * NeAUT * NeX) / (9 * NeAUT - 8 * NeX)
    else:
        raise ValueError("Invalid keys in dNe. Expected combinations: ['MT', 'Y'], ['MT', 'AUT'], ['MT', 'X'], ['AUT', 'X']")
    return {"MT": NeMT, "Y": NeY, "X": NeX, "AUT": NeAUT}

In [12]:
def calc_direct_Ne(theta_pop, mu_coef, mu_AUT):
    Ne = {}
    for comp in COMP_ORDER:
        ploidy_factor = 2 if comp in ["AUT", "X"] else 1
        Ne[comp] = theta_pop[comp] / (2 * ploidy_factor * mu_coef[comp] * mu_AUT)
    return Ne

In [13]:
def build_pairwise_reconstructions(Ne_direct):
    out = {}
    for pair in PAIR_KEYS:
        pair_dict = {k: Ne_direct[k] for k in pair}
        out["_".join(pair)] = get_Ne_per_component(pair_dict)
    return out

In [14]:
def summarize_population(pop, theta_pop, mu_coef, mu_AUT):
    Ne_direct = calc_direct_Ne(theta_pop, mu_coef, mu_AUT)
    recon = build_pairwise_reconstructions(Ne_direct)

    df = pd.DataFrame(index=COMP_ORDER)

    for pair_name, vals in recon.items():
        df[pair_name] = [vals[c] for c in COMP_ORDER]

    df["theta_direct"] = [Ne_direct[c] for c in COMP_ORDER]

    return df

# Point estimates of Ne per component with a fixed mutation rate

In [15]:
# mu_coef = {"MT": 10, "Y": 1.7, "X": 0.8, "AUT": 1}
mu_coef = {
   'MT': 10,
   'Y': 1.5,
   'X': 0.7,
   'AUT': 1
}

for sp in theta.keys():
  print(f"\n=== {sp} ===")
  display(summarize_population(sp, theta[sp], mu_coef, mu_AUT).round(6))


=== GSL ===


,MT_Y,MT_AUT,MT_X,AUT_X,theta_direct
MT,3792.857143,3792.857143,3792.857143,2891.456661,3792.857143
Y,3000.000000,8814.386318,5111.328526,31991.877097,3000.000000
X,5228.665208,7022.933737,6224.489796,6224.489796,6224.489796
AUT,6700.315457,10607.142857,8708.955374,10607.142857,10607.142857



=== SSL ===


,MT_Y,MT_AUT,MT_X,AUT_X,theta_direct
MT,7785.714286,7785.714286,7785.714286,4721.187404,7785.714286
Y,5333.333333,9712.076583,5366.668316,51036.456125,5333.333333
X,10126.451613,12505.372946,10153.061224,10153.061224,10153.061224
AUT,12660.617060,17285.714286,12707.460675,17285.714286,17285.714286


# Optimisation of point estimates by mutation rate ranges

In [16]:
# sp = "SSL"
# theta = {sp: theta[sp]} 
rmu_coef = {
    "MT": [10, 20],
    "Y": [1.5, 2.5],
    "X": [0.7, 0.85],
    "AUT": [1, 1],
}

def objective(params, theta, mu_AUT, use_log_error=False, verbose=False):
    mu_coef = {
        "MT": params[0],
        "Y": params[1],
        "X": params[2],
        "AUT": 1.0,
    }

    total_error = 0.0
    penalty = 1e12

    for pop, theta_pop in theta.items():
        try:
            Ne_direct = calc_direct_Ne(theta_pop, mu_coef, mu_AUT)
            recon = build_pairwise_reconstructions(Ne_direct)

            for pair_name, rec_vals in recon.items():
                for comp in COMP_ORDER:
                    target = Ne_direct[comp]
                    pred = rec_vals[comp]

                    if target <= 0 or pred <= 0 or not np.isfinite(target) or not np.isfinite(pred):
                        return penalty

                    if use_log_error:
                        err = (np.log(pred) - np.log(target)) ** 2
                    else:
                        err = ((pred - target) / target) ** 2

                    total_error += err

        except Exception:
            return penalty

    return total_error

# run optimization
bounds = [
    tuple(rmu_coef["MT"]),
    tuple(rmu_coef["Y"]),
    tuple(rmu_coef["X"]),
]

result = differential_evolution(
    objective,
    bounds=bounds,
    args=(theta, mu_AUT, True),
    seed=42,
    polish=True,
    tol=1e-8,
)

best_mu_coef = {
    "MT": result.x[0],
    "Y": result.x[1],
    "X": result.x[2],
    "AUT": 1.0,
}

print("Optimization success:", result.success)
print("Best mu_coef:")
for k, v in best_mu_coef.items():
    print(f"  {k}: {v:.6f}")

# for pop, theta_pop in theta.items():
#     print(f"\n=== {pop} ===")
#     display(summarize_population(pop, theta_pop, best_mu_coef, mu_AUT).round(6))

Optimization success: True
Best mu_coef:
  MT: 10.000000
  Y: 1.500000
  X: 0.700000
  AUT: 1.000000
